Training and testing the model used in NeuroDetect

## Imports

In [23]:
import tensorflow as tf
import os
import shutil
import random
import numpy as np
from pathlib import Path
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

## Constants


In [24]:
IMG_SIZE = 256
BATCH_SIZE = 32

In [25]:
SRC_DIR = Path(r"D:\GDrive\SeniorProject\datasets\dataset\train")
OUT_DIR = Path(r"D:\GDrive\SeniorProject\datasets\bt_split")

TRAIN_RAT = 0.70
VAL_RAT =0.15
TEST_RAT=0.15

random.seed(42)


if (OUT_DIR / "train").exists():
    print("dataset already split. skipping.")

else:
    print("create split..")

    for class_folder in SRC_DIR.iterdir():
        if not class_folder.is_dir():
            continue
        
        images = list(class_folder.glob("*.*"))
        random.shuffle(images)

        total = len(images)
        train = int(total * TRAIN_RAT)
        val = train + int(total * VAL_RAT)

        split_data = {
            "train": images[:train],
            "val": images[train:val],
            "test": images[val:]
        }

        for split_name, files in split_data.items():
            dest = OUT_DIR / split_name / class_folder.name
            dest.mkdir(parents=True, exist_ok=True)
            for f in files:
                shutil.copy(f, dest / f.name)
            print( class_folder.name, "train:", len(split_data["train"]), "val:", len(split_data["val"]), "test:", len(split_data["test"]))

    print("done✔️")

dataset already split. skipping.


## Paths

In [26]:

PROJECT_DIR = Path(r"D:\GDrive\SeniorProject\datasets\bt_split")

TRAIN_DIR = str(PROJECT_DIR / "train")
VAL_DIR   = str(PROJECT_DIR / "val")
TEST_DIR  = str(PROJECT_DIR / "test")

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True, parents=True)

for n, d in[("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
    print(f"{n}: {"exists" if os.path.exists(d) else 'Missing'}")

train: exists
val: exists
test: exists


#### Done with Baseline Model training ↓

# Baseline Model

In [27]:
# num_classes = len(labels)

# model_baseline = tf.keras.Sequential([
#     tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Flatten(),

#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dropout(0.5),

#     tf.keras.layers.Dense(num_classes, activation='softmax')
# ])

# model_baseline.summary()

### Compile

In [28]:
# model_baseline.compile(
#     optimizer='adam',
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

### Run Epochs

In [29]:
# import time
# import math

# steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
# validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

# start = time.time()
# history_baseline = model_baseline.fit(
#     train_generator,
#     steps_per_epoch=steps_per_epoch,
#     validation_data=test_generator,
#     validation_steps=validation_steps,
#     epochs=EPOCHS
# )
# train_time_sec = time.time() - start

# print("Training time (sec):", round(train_time_sec, 2))

### Results

In [30]:
# baseline_loss, baseline_acc = model_baseline.evaluate(
#     test_generator,
#     steps=validation_steps
# )
# print("Baseline Loss:", baseline_loss)
# print("Baseline Accuracy:", baseline_acc)

### Analysis

In [31]:
# import numpy as np
# from sklearn.metrics import confusion_matrix, classification_report

# # reset generator to start
# test_generator.reset()

# y_prob = model_baseline.predict(test_generator, steps=validation_steps)
# y_pred = np.argmax(y_prob, axis=1)

# y_true = test_generator.classes  # integer labels in directory order
# class_names = list(test_generator.class_indices.keys())

# cm = confusion_matrix(y_true, y_pred)
# print("Confusion Matrix:\n", cm)

# print("\nClassification Report:\n")
# print(classification_report(y_true, y_pred, target_names=class_names))

### Save Model

In [32]:
# os.makedirs(MODELS_DIR, exist_ok=True)

# baseline_model_path = MODELS_DIR / "baseline_cnn.keras"
# model_baseline.save(str(baseline_model_path))
# print(f"Saved to {baseline_model_path}")

# Transfer Model

### Generators

In [33]:
transfer_train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=8,
    width_shift_range=0.04,
    height_shift_range=0.04,
    zoom_range=0.08,
    horizontal_flip=True,
    brightness_range=[0.95, 1.05],
    fill_mode='nearest'
)

transfer_eval_datagen = ImageDataGenerator()

train_gen_tf = transfer_train_datagen.flow_from_directory(TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)

val_gen_tf = transfer_eval_datagen.flow_from_directory(VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical',shuffle=False)

test_gen_tf = transfer_eval_datagen.flow_from_directory(TEST_DIR,target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,class_mode='categorical',shuffle=False)

print("class labels:",train_gen_tf.class_indices)
print("generators ready")

Found 5929 images belonging to 4 classes.
Found 1269 images belonging to 4 classes.
Found 1272 images belonging to 4 classes.
class labels: {'glioma_tumor': 0, 'meningioma_tumor': 1, 'no_tumor': 2, 'pituitary_tumor': 3}
generators ready


## Phase 1

### Model Build + Training
##### Skips training if model is saved already

In [34]:
PHASE1_PATH = MODEL_DIR / "phase1.keras"

if PHASE1_PATH.exists():
    print("Phase 1 already trained. loading model..")
    model_transfer = tf.keras.models.load_model(str(PHASE1_PATH))
else:
    print("training Phase 1..")

    base_model = EfficientNetB3(weights='imagenet',include_top=False,input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.6)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(train_gen_tf.num_classes, activation='softmax')(x)

    model_transfer = Model(inputs=base_model.input, outputs=outputs)
    model_transfer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),metrics=['accuracy'])

    history_phase1 = model_transfer.fit(
        train_gen_tf,
        validation_data=val_gen_tf,
        epochs=20,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max'),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
            ModelCheckpoint(str(PHASE1_PATH), monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
        ]
    )

    model_transfer.save(str(PHASE1_PATH))
    print("Saved Phase 1")

phase1_loss, phase1_acc = model_transfer.evaluate(val_gen_tf)
print(f"Phase 1 Validation Accuracy: {phase1_acc:.4f}")

Phase 1 already trained. loading model..
40/40 ━━━━━━━━━━━━━━━━━━━━ 67s 2s/step - accuracy: 0.9488 - loss: 0.3469
Phase 1 Validation Accuracy: 0.9488


## Phase 2

### Fine Tune

In [35]:
PHASE2_PATH = MODEL_DIR / "phase2.keras"

if PHASE2_PATH.exists():
    print("Phase 2 already trained. loading model..")
    model_transfer = tf.keras.models.load_model(str(PHASE2_PATH))

else:
    print("training Phase 2...")

    for layer in model_transfer.layers:
        layer.trainable = False

    for layer in model_transfer.layers[-40:]:
        layer.trainable = True

    for layer in model_transfer.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    trainable_count = sum(1 for layer in model_transfer.layers if layer.trainable)
    print("Trainable layers:", trainable_count)

    model_transfer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),metrics=['accuracy'])


    history_phase2 = model_transfer.fit(
        train_gen_tf,
        validation_data=val_gen_tf,
        epochs=25,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, mode='max'),
            ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-8, verbose=1),
            ModelCheckpoint(str(PHASE2_PATH), monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
        ]
    )

    model_transfer.save(str(PHASE2_PATH))
    print("Saved Phase 2")

phase2_loss, phase2_acc = model_transfer.evaluate(val_gen_tf)
print(f"Phase 2 Validation Accuracy: {phase2_acc:.4f}")

Phase 2 already trained. loading model..
40/40 ━━━━━━━━━━━━━━━━━━━━ 60s 1s/step - accuracy: 0.9913 - loss: 0.2331
Phase 2 Validation Accuracy: 0.9913


In [41]:
model = tf.keras.models.load_model(
    r"D:\GDrive\SeniorProject\datasets\bt_split\models\phase2.keras",
    compile=False
)
model.save("phase2_clean.keras")  # save as .keras not .h5
print("Done")

Done


## Evaluate Improvements

### Final Test Evaluation

In [ ]:
test_gen_tf.reset()
test_loss, test_acc = model_transfer.evaluate(test_gen_tf)
print(f"Testing Accuracy: {test_acc:.4f}")

y_pred = np.argmax(model_transfer.predict(test_gen_tf), axis=1)
y_true = test_gen_tf.classes
class_names = list(test_gen_tf.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\n───────────── Summary ────────────")
print(f"Phase 1 Val Accuracy: {phase1_acc:.4f}")
print(f"Phase 2 Val Accuracy: {phase2_acc:.4f}")
print(f"Fin Accuracy: {test_acc:.4f}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.9882 - loss: 0.2391
Final Test Accuracy: 0.9882
40/40 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step
Confusion Matrix:
 [[334   1   0   0]
 [  3 323   2   6]
 [  0   1 268   0]
 [  0   2   0 332]]

Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       0.99      1.00      0.99       335
meningioma_tumor       0.99      0.97      0.98       334
        no_tumor       0.99      1.00      0.99       269
 pituitary_tumor       0.98      0.99      0.99       334

        accuracy                           0.99      1272
       macro avg       0.99      0.99      0.99      1272
    weighted avg       0.99      0.99      0.99      1272


── Summary ──
Phase 1 Validation Accuracy: 0.9488
Phase 2 Validation Accuracy: 0.9913
Final Test Accuracy: 0.9882
